<a href="https://colab.research.google.com/github/grsart/BiomolComp/blob/main/Pratica6/AnaliseFilogenetica3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise Filogenética de Sequências – Parte 3 (Python) #

Também podemos usar ferramentas do [Biopython](http://biopython.org) e programas de linha de comando para análises filogenéticas.

Nesta parte vamos repetir, em Python, a análise feita no MEGA na Parte 1, e comparar três árvores construídas com as mesmas sequências:

1. a **árvore-guia** que o ClustalW gera para construir o alinhamento (que **não** é uma árvore filogenética);
2. uma **árvore filogenética por Neighbor-Joining** com bootstrap, equivalente à da Parte 1;
3. *(opcional)* uma **árvore de máxima verossimilhança** com o IQ-TREE.

> **Tempo estimado:** 40 minutos (a seção opcional leva mais 5 minutos). Esta parte também pode ser feita como tarefa de casa.  
> **O que entregar:** respostas das questões Q16 a Q18 (e Q19, se fizer a seção opcional).

Execute as células na ordem, de cima para baixo.

In [ ]:
#@title Preparando o ambiente para a execução

# Biopython com versão fixa, para que o notebook continue funcionando
# mesmo quando novas versões mudarem a interface da biblioteca
!pip3 install -q biopython==1.85

# ClustalW (alinhamento) e IQ-TREE (máxima verossimilhança, seção opcional)
!apt-get install -y -qq clustalw iqtree > /dev/null

import os
import shutil
import subprocess
import urllib.request

import matplotlib.pyplot as plt
from Bio import SeqIO, AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Phylo.Consensus import bootstrap_trees, get_support

assert shutil.which("clustalw"), "ClustalW não foi instalado"
print("Ambiente pronto.")

# Análise Filogenética #

Nesta seção, vamos fazer o download das sequências do domínio transcriptase reversa de retrotransposons do tipo non-LTR que já usamos na Parte 1.

Na sequência inspecionaremos as sequências presentes neste arquivo.

In [ ]:
seq_file = "filogenia1.fasta"
url = "https://raw.githubusercontent.com/grsart/BiomolComp/main/Pratica6/filogenia1.fasta"
urllib.request.urlretrieve(url, seq_file)

records = list(SeqIO.parse(seq_file, "fasta"))
print(f"{'SeqID':<12} {'Tamanho':>8}")
for rec in records:
    print(f"{rec.id:<12} {len(rec):>8}")
print(f"\nNúmero de sequências: {len(records)}")

## Alinhamento das Sequências ##

Nesta etapa, usamos o programa [ClustalW](http://www.clustal.org/clustal2/) para preparar um alinhamento múltiplo de sequências (MSA), com as mesmas penalidades de gap usadas no MEGA (abertura 5.0, extensão 0.1). Como o ClustalW utiliza o método progressivo para o alinhamento múltiplo, as distâncias entre as sequências já são calculadas e salvas juntamente com o alinhamento. Efetivamente, o programa gera um arquivo:

```
to store a dendrogram that describes the similarity
of the sequences to each other.
```

Este arquivo **não é uma árvore filogenética**. Ele é a árvore guia (*guide-tree*) para a construção do alinhamento múltiplo. Mas, como o texto acima nos diz, ele agrupa as sequências similares, com as distâncias entre elas. Isto pode dar uma ideia do quão próximas as sequências são dentro de um alinhamento múltiplo.

Na etapa abaixo, vamos fazer o alinhamento múltiplo (MSA) com o ClustalW, chamando o programa diretamente pela linha de comando.

In [ ]:
result = subprocess.run(
    ["clustalw", f"-INFILE={seq_file}", "-GAPOPEN=5", "-GAPEXT=0.1"],
    capture_output=True, text=True)
assert result.returncode == 0, result.stdout + result.stderr

align = AlignIO.read("filogenia1.aln", "clustal")
print(f"Alinhamento com {len(align)} sequências e {align.get_alignment_length()} colunas\n")

# Mostra um trecho do alinhamento em torno do motivo FADD (YADD) da transcriptase reversa
inicio = str(align[0].seq).find("FADD")
trecho = align[:, max(inicio - 40, 0):inicio + 20]
for record in trecho:
    print(f"{record.id:<12} {record.seq}")

## Análise da Árvore Guia ##

O programa ClustalW gerou dois arquivos. O arquivo filogenia1.aln contém o alinhamento gerado pelo método progressivo (visualizado na célula acima).

Já o arquivo filogenia1.dnd contém o dendrograma com as distâncias (ou similaridades) que as sequências têm entre si. Ele está no formato Newick e pode ser lido diretamente pelo Biopython.

No desenho em texto (ASCII) abaixo, o comprimento dos ramos é proporcional à distância entre as sequências.

In [ ]:
guide_tree = Phylo.read("filogenia1.dnd", "newick")
Phylo.draw_ascii(guide_tree)

## Árvore filogenética por Neighbor-Joining ##

Agora vamos construir uma árvore filogenética de verdade, com o mesmo método usado no MEGA (Neighbor-Joining), a partir do **alinhamento** e não da árvore-guia:

* `DistanceCalculator("blosum62")` calcula a distância entre cada par de sequências alinhadas usando a matriz BLOSUM62;
* `DistanceTreeConstructor(..., "nj")` constrói a árvore pelo método Neighbor-Joining;
* `bootstrap_trees` reamostra as colunas do alinhamento e reconstrói a árvore várias vezes, e `get_support` calcula, para cada ramo, a porcentagem de réplicas em que ele aparece.

Assim como no MEGA, a árvore NJ não tem raiz. Vamos enraizá-la usando o CRE2 como grupo externo.

**Atenção:** com 100 réplicas, a célula abaixo leva de 1 a 3 minutos.

In [ ]:
n_replicas = 100

calculator = DistanceCalculator("blosum62")
constructor = DistanceTreeConstructor(calculator, "nj")

nj_tree = constructor.build_tree(align)
replicas = list(bootstrap_trees(align, n_replicas, constructor))
nj_tree = get_support(nj_tree, replicas)

# get_support já devolve o suporte em porcentagem; removemos os nomes
# automáticos dos nós internos ("Inner1", "Inner2"...)
for clade in nj_tree.get_nonterminals():
    clade.name = None
    if clade.confidence is not None:
        clade.confidence = round(clade.confidence)

nj_tree.root_with_outgroup("CRE2")
# O ramo da raiz separa o CRE2 de todo o resto: o suporte dele não é informativo
for clade in nj_tree.root.clades:
    clade.confidence = None
nj_tree.ladderize()
print("Árvore NJ construída e enraizada no CRE2.")

## Comparação: árvore-guia × árvore NJ ##

As duas árvores são desenhadas lado a lado. Na árvore NJ, os números nos ramos são os valores de bootstrap (%).

Observe que a árvore-guia **não tem raiz definida nem valores de suporte**: ela é desenhada a partir de um ponto arbitrário.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 11))

Phylo.draw(guide_tree, axes=axes[0], do_show=False)
axes[0].set_title("Árvore-guia do ClustalW (filogenia1.dnd)")

Phylo.draw(nj_tree, axes=axes[1], do_show=False,
           branch_labels=lambda c: c.confidence if c.confidence is not None and not c.is_terminal() else None)
axes[1].set_title(f"Neighbor-Joining (BLOSUM62), bootstrap {n_replicas} réplicas, raiz no CRE2")

plt.tight_layout()
plt.show()

Compare a árvore-guia com a árvore NJ acima e com a árvore obtida no MEGA na Parte 1.

**Q16- A árvore-guia e a árvore NJ são similares? Explique considerando as diferenças entre uma árvore-guia e uma árvore filogenética.**

**Q17- Por que não é seguro fazer inferências filogenéticas a partir de uma árvore-guia?**

**Q18- A árvore NJ deste notebook classifica z82275, pido e Sr1 na mesma família que a árvore do MEGA? Os valores de bootstrap são parecidos? Por que eles podem variar um pouco entre as duas análises?** *(Dica: compare a matriz de distâncias, BLOSUM62 aqui e JTT no MEGA, e lembre que o bootstrap é um processo aleatório.)*

# *(Opcional)* Máxima verossimilhança com o IQ-TREE #

Os métodos de distância (como o NJ) resumem o alinhamento em uma única matriz de distâncias. Os métodos de **máxima verossimilhança** (*maximum likelihood*, ML) usam cada coluna do alinhamento e um modelo explícito de substituição de aminoácidos.

O [IQ-TREE](http://www.iqtree.org/) é um dos programas de ML mais usados. O comando abaixo:

* `-m MFP -mset LG,JTT,WAG`: usa o **ModelFinder** para escolher, por critério estatístico (BIC), o melhor modelo de substituição entre LG, JTT e WAG (com ou sem variação de taxas entre sítios);
* `-B 1000`: calcula o suporte dos ramos por **ultrafast bootstrap** (UFBoot) com 1000 réplicas.

**Atenção:** o UFBoot não tem a mesma escala do bootstrap clássico. Para UFBoot, considera-se bom suporte valores **≥ 95%**.

A célula leva de 1 a 3 minutos.

In [ ]:
AlignIO.write(align, "filogenia1_aln.fasta", "fasta")

iqtree_exe = shutil.which("iqtree2") or shutil.which("iqtree")
result = subprocess.run(
    [iqtree_exe, "-s", "filogenia1_aln.fasta", "-m", "MFP", "-mset", "LG,JTT,WAG",
     "-B", "1000", "-T", "AUTO", "--prefix", "ml", "-redo", "-quiet"],
    capture_output=True, text=True)
assert result.returncode == 0, result.stdout + result.stderr

with open("ml.iqtree") as f:
    for line in f:
        if line.startswith("Best-fit model"):
            print(line.strip())

ml_tree = Phylo.read("ml.treefile", "newick")
ml_tree.root_with_outgroup("CRE2")
for clade in ml_tree.root.clades:
    clade.confidence = None
ml_tree.ladderize()

fig, ax = plt.subplots(figsize=(9, 11))
Phylo.draw(ml_tree, axes=ax, do_show=False,
           branch_labels=lambda c: c.confidence if c.confidence is not None and not c.is_terminal() else None)
ax.set_title("Máxima verossimilhança (IQ-TREE), UFBoot 1000 réplicas, raiz no CRE2")
plt.tight_layout()
plt.show()

**Q19 (opcional)- Qual modelo de substituição o ModelFinder escolheu? Compare a árvore de máxima verossimilhança com a árvore NJ: as famílias são as mesmas? Em quais ramos as duas árvores discordam, e qual é o suporte desses ramos em cada uma?**